In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("../data/European_Bank_Processed.csv")

print("Dataset shape:", df.shape)

Dataset shape: (10000, 16)


In [3]:
# Overall KPIs

total_customers = len(df)

total_churners = df["Exited"].sum()

overall_churn_rate = (
    total_churners / total_customers * 100
)

retention_rate = 100 - overall_churn_rate

high_value_customers = (
    df["BalanceSegment"] == "High-balance"
).sum()

high_value_churners = (
    (df["BalanceSegment"] == "High-balance") &
    (df["Exited"] == 1)
).sum()

high_value_churn_rate = (
    high_value_churners /
    high_value_customers * 100
)

high_value_exposure = df.loc[
    (df["BalanceSegment"] == "High-balance") &
    (df["Exited"] == 1),
    "Balance"
].sum()

print("Total Customers:", total_customers)
print("Total Churners:", total_churners)
print("Overall Churn Rate:", round(overall_churn_rate, 2), "%")
print("Retention Rate:", round(retention_rate, 2), "%")
print("High-Value Customers:", high_value_customers)
print("High-Value Churners:", high_value_churners)
print("High-Value Churn Rate:", round(high_value_churn_rate, 2), "%")
print(
    "High-Value Financial Exposure: €",
    round(high_value_exposure, 2)
)

Total Customers: 10000
Total Churners: 2037
Overall Churn Rate: 20.37 %
Retention Rate: 79.63 %
High-Value Customers: 5000
High-Value Churners: 1249
High-Value Churn Rate: 24.98 %
High-Value Financial Exposure: € 163242872.31


Geographic insight 

In [4]:
geography_summary = (
    df.groupby("Geography")["Exited"]
    .agg(["count", "sum", "mean"])
    .reset_index()
)

geography_summary["ChurnRate"] = (
    geography_summary["mean"] * 100
)

geography_summary["ChurnContribution"] = (
    geography_summary["sum"] /
    total_churners * 100
)

geography_summary = (
    geography_summary
    .drop(columns="mean")
    .sort_values("ChurnRate", ascending=False)
)

geography_summary

,Geography,count,sum,ChurnRate,ChurnContribution
1,Germany,2509,814,32.443204,39.960727
2,Spain,2477,413,16.673395,20.274914
0,France,5014,810,16.154767,39.764359


Age insight

In [5]:
age_summary = (
    df.groupby("AgeGroup")["Exited"]
    .agg(["count", "sum", "mean"])
    .reset_index()
)

age_summary["ChurnRate"] = (
    age_summary["mean"] * 100
)

age_summary["ChurnContribution"] = (
    age_summary["sum"] /
    total_churners * 100
)

age_summary = (
    age_summary
    .drop(columns="mean")
    .sort_values("ChurnRate", ascending=False)
)

age_summary

,AgeGroup,count,sum,ChurnRate,ChurnContribution
1,46-60,1647,842,51.123254,41.335297
2,60+,464,115,24.784483,5.645557
0,30-45,6248,956,15.300896,46.931762
3,<30,1641,124,7.556368,6.087383


Engagement insight

In [6]:
engagement_summary = (
    df.groupby("EngagementStatus")["Exited"]
    .agg(["count", "sum", "mean"])
    .reset_index()
)

engagement_summary["ChurnRate"] = (
    engagement_summary["mean"] * 100
)

engagement_summary["ChurnContribution"] = (
    engagement_summary["sum"] /
    total_churners * 100
)

engagement_summary = (
    engagement_summary
    .drop(columns="mean")
    .sort_values("ChurnRate", ascending=False)
)

engagement_summary

,EngagementStatus,count,sum,ChurnRate,ChurnContribution
1,Inactive,4849,1302,26.850897,63.917526
0,Active,5151,735,14.269074,36.082474


Strongest combined risk profile

In [7]:
risk_profile = (
    df.groupby(
        [
            "Geography",
            "AgeGroup",
            "EngagementStatus",
            "BalanceSegment"
        ]
    )["Exited"]
    .agg(["count", "sum", "mean"])
    .reset_index()
)

risk_profile["ChurnRate"] = (
    risk_profile["mean"] * 100
)

risk_profile["ChurnContribution"] = (
    risk_profile["sum"] /
    total_churners * 100
)

risk_profile = risk_profile.drop(columns="mean")

reliable_risk_profile = (
    risk_profile[
        risk_profile["count"] >= 50
    ]
    .sort_values(
        "ChurnRate",
        ascending=False
    )
)

reliable_risk_profile.head(10)

,Geography,AgeGroup,EngagementStatus,BalanceSegment,count,sum,ChurnRate,ChurnContribution
30,Germany,46-60,Inactive,High-balance,237,195,82.278481,9.572901
49,Spain,46-60,Inactive,High-balance,64,40,62.500000,1.963672
9,France,46-60,Inactive,High-balance,144,87,60.416667,4.270987
11,France,46-60,Inactive,Zero-balance,154,92,59.740260,4.516446
51,Spain,46-60,Inactive,Zero-balance,76,45,59.210526,2.209131
28,Germany,46-60,Active,High-balance,195,105,53.846154,5.154639
6,France,46-60,Active,High-balance,144,51,35.416667,2.503682
26,Germany,30-45,Inactive,High-balance,616,212,34.415584,10.407462
7,France,46-60,Active,Low-balance,63,21,33.333333,1.030928
44,Spain,30-45,Inactive,Low-balance,93,30,32.258065,1.472754


In [8]:
balance_summary = (
    df.groupby("BalanceSegment")["Exited"]
    .agg(["count", "sum", "mean"])
    .reset_index()
)

balance_summary["ChurnRate"] = (
    balance_summary["mean"] * 100
)

balance_summary["ChurnContribution"] = (
    balance_summary["sum"] /
    total_churners * 100
)

balance_summary = (
    balance_summary
    .drop(columns="mean")
    .sort_values("ChurnRate", ascending=False)
)

balance_summary

,BalanceSegment,count,sum,ChurnRate,ChurnContribution
0,High-balance,5000,1249,24.980000,61.315660
1,Low-balance,1383,288,20.824295,14.138439
2,Zero-balance,3617,500,13.823611,24.545901


In [9]:
final_kpis = pd.DataFrame({
    "KPI": [
        "Overall Churn Rate",
        "Germany Churn Rate",
        "Age 46-60 Churn Rate",
        "Inactive Customer Churn Rate",
        "High-Value Churn Rate",
        "High-Value Churn Contribution",
        "High-Value Financial Exposure",
        "Highest-Risk Combined Segment Churn Rate"
    ],
    "Value": [
        overall_churn_rate,
        geography_summary.loc[
            geography_summary["Geography"] == "Germany",
            "ChurnRate"
        ].iloc[0],
        age_summary.loc[
            age_summary["AgeGroup"] == "46-60",
            "ChurnRate"
        ].iloc[0],
        engagement_summary.loc[
            engagement_summary["EngagementStatus"] == "Inactive",
            "ChurnRate"
        ].iloc[0],
        high_value_churn_rate,
        high_value_churners / total_churners * 100,
        high_value_exposure,
        reliable_risk_profile.iloc[0]["ChurnRate"]
    ]
})

final_kpis["Value"] = final_kpis["Value"].round(2)

final_kpis

,KPI,Value
0,Overall Churn Rate,2.037000e+01
1,Germany Churn Rate,3.244000e+01
2,Age 46-60 Churn Rate,5.112000e+01
3,Inactive Customer Churn Rate,2.685000e+01
4,High-Value Churn Rate,2.498000e+01
5,High-Value Churn Contribution,6.132000e+01
6,High-Value Financial Exposure,1.632429e+08
7,Highest-Risk Combined Segment Churn Rate,8.228000e+01


# Final Key Findings

## 1. Overall Churn
The dataset contains 10,000 customers, of which 2,037 have churned. The overall observed churn rate is 20.37%, while 79.63% of customers have been retained.

## 2. Geographic Churn Risk
Germany shows the highest observed churn rate at 32.44%, substantially higher than France (16.15%) and Spain (16.67%). Germany also contributes approximately 39.96% of all churners despite representing only 25.09% of the customer base.

## 3. Age-Based Churn Pattern
Customers aged 46–60 represent the strongest observed age-related churn segment, with a churn rate of 51.12%. This group accounts for approximately 41.34% of all churners.

## 4. Customer Engagement
Inactive customers show a churn rate of 26.85%, compared with 14.27% among active customers. Inactive customers represent 48.49% of the customer base but account for approximately 63.92% of all churners, indicating a strong association between inactivity and churn.

## 5. High-Value Customer Churn
High-value customers are defined in this analysis as customers belonging to the High-balance segment. There are 5,000 such customers, of whom 1,249 have churned, resulting in a 24.98% churn rate.

These high-value churners represent approximately 61.32% of all churners.

## 6. Financial Exposure
The total balance associated with churned high-value customers is approximately €163.24 million. This represents financial exposure associated with customers who have already churned and should not be interpreted as direct lost revenue because the dataset does not contain revenue or profit information.

## 7. Combined Risk Profile
The highest observed churn rate among sufficiently sized combined segments occurs for customers in Germany who are aged 46–60, inactive, and have a high balance.

This segment contains 237 customers, of whom 195 churned, resulting in an observed churn rate of 82.28%.

## 8. Overall Insight
The analysis indicates that churn is not evenly distributed across the customer base. Geography, age, engagement status, and balance segment provide stronger segmentation signals than several individual financial or demographic variables.

The findings represent observed associations in the dataset and should not be interpreted as evidence of causal relationships.

# Strategic Recommendations

## 1. Prioritize Customer Retention in Germany
Germany has the highest observed churn rate at 32.44% and contributes approximately 39.96% of all churners.

### Recommended Action
Develop a Germany-focused retention strategy that includes:
- Early identification of customers showing disengagement.
- Targeted retention communication.
- Review of customer experience and service issues.
- Monitoring churn trends within German customer segments.

---

## 2. Develop a Dedicated Strategy for Customers Aged 46–60
The 46–60 age group has the highest observed churn rate at 51.12% and contributes approximately 41.34% of all churners.

### Recommended Action
Create a dedicated retention program for this segment, particularly when combined with other risk indicators such as:
- Germany
- Inactive membership
- High balance

---

## 3. Re-Engage Inactive Customers
Inactive customers have an observed churn rate of 26.85%, compared with 14.27% for active customers. They account for approximately 63.92% of all churners.

### Recommended Action
Implement customer re-engagement initiatives such as:
- Personalized communication.
- Product and service reminders.
- Relevant offers based on customer profiles.
- Regular monitoring of inactivity.

The objective should be to identify disengagement before it develops into customer churn.

---

## 4. Protect High-Value Customers
High-balance customers have a churn rate of 24.98% and account for approximately 61.32% of all churners.

The balance associated with churned high-value customers represents approximately €163.24 million in financial exposure.

### Recommended Action
Establish a high-value customer monitoring program that:
- Tracks churn risk among high-balance customers.
- Prioritizes high-value customers showing inactivity.
- Provides proactive relationship management.
- Monitors high-value churn by geography and age.

---

## 5. Prioritize Combined Risk Profiles
The highest observed churn rate occurs among customers who are:

**Germany + Age 46–60 + Inactive + High-balance**

This segment has an observed churn rate of 82.28% across 237 customers.

### Recommended Action
Use a multi-factor segmentation approach instead of relying on a single characteristic.

Customers showing multiple risk indicators should receive higher retention priority.

---

## 6. Use Data-Driven Customer Segmentation
The analysis shows that churn is unevenly distributed across customer groups.

### Recommended Action
The bank should monitor customers using multiple dimensions, including:
- Geography
- Age group
- Engagement status
- Balance segment
- Credit score
- Tenure
- Number of products

This approach can help identify specific customer groups that require closer attention.

---

## 7. Monitor Churn KPIs Regularly
The identified KPIs should be tracked over time rather than evaluated only once.

### Recommended Action
Create a recurring churn monitoring process covering:
- Overall Churn Rate
- Geographic Churn Rate
- Segment Churn Rate
- High-Value Churn Rate
- High-Value Churn Contribution
- Financial Exposure
- Engagement-related churn indicators

Regular monitoring can help stakeholders identify changes in customer behavior and evaluate the effectiveness of retention initiatives.

---

## Strategic Priority

Based on the analysis, the recommended priority order is:

1. **Germany-focused retention**
2. **46–60 age-group retention**
3. **Inactive customer re-engagement**
4. **High-value customer protection**
5. **Combined-risk segment monitoring**
6. **Continuous KPI monitoring**

These recommendations are based on observed patterns in the dataset. They should be validated with additional customer feedback, operational data, revenue information, and longitudinal churn data before being used to make final business decisions.

# Project Limitations

Although the analysis provides useful customer segmentation and churn insights, several limitations should be considered.

## 1. Observational Analysis
The analysis identifies associations between customer characteristics and churn. It does not establish causal relationships.

For example, the higher churn observed among inactive customers does not prove that inactivity directly causes churn.

## 2. No Direct Revenue or Profit Data
The dataset does not contain actual revenue, profit, customer lifetime value, or product profitability information.

Therefore, the €163.24 million figure should be interpreted as financial exposure represented by the balance of churned high-value customers, rather than confirmed lost revenue.

## 3. Limited Time Information
The dataset provides customer attributes but does not provide a detailed longitudinal record of customer behavior over time.

Therefore, changes in customer behavior before and after churn cannot be directly analyzed.

## 4. Segment Size Differences
Some segments are considerably smaller than others.

For example, the 3-product and 4-product groups contain relatively few customers. Their high observed churn rates should therefore be interpreted cautiously.

## 5. No External Customer Feedback
The dataset does not include information such as:
- Customer satisfaction
- Complaints
- Service quality
- Reasons for leaving
- Customer support interactions

These factors could provide additional explanations for observed churn patterns.

## 6. Limited Geographic Coverage
The dataset contains customers from France, Germany, and Spain only. Therefore, the findings should not automatically be generalized to other countries or banking markets.

## 7. Segmentation Thresholds
Age, credit score, tenure, and balance groups were created using predefined thresholds for analytical purposes.

Different segmentation boundaries could produce somewhat different segment-level results.

## 8. Need for Further Validation
The identified high-risk segments should be validated using additional operational and behavioral data before being used for automated customer-retention decisions.

# Final Conclusion

This project analyzed customer churn patterns in a European banking dataset containing 10,000 customers. Through data validation, exploratory analysis, customer segmentation, churn analysis, and high-value customer analysis, several important patterns were identified.

The overall observed churn rate is 20.37%. Germany represents the highest geographic churn risk, with a churn rate of 32.44%. Customers aged 46–60 show the highest age-based churn rate at 51.12%, while inactive customers have a churn rate of 26.85% compared with 14.27% among active customers.

High-value customers, defined as customers in the High-balance segment, show a churn rate of 24.98% and account for approximately 61.32% of all churners. The balance associated with churned high-value customers represents approximately €163.24 million in financial exposure.

The most critical combined segment identified in the analysis is Germany + 46–60 + Inactive + High-balance, which has an observed churn rate of 82.28% across 237 customers. This demonstrates the value of combining multiple customer characteristics when identifying groups requiring retention attention.

Overall, the analysis suggests that customer churn is unevenly distributed across the customer base. Geography, age, engagement status, and balance segment provide useful signals for customer segmentation and retention planning.

The findings can support targeted retention strategies, customer re-engagement initiatives, and high-value customer monitoring. However, the identified relationships represent observed associations rather than causal relationships and should be validated with additional behavioral, operational, customer-feedback, and financial data.

The completed analysis provides a foundation for the next stage of the project: presenting these findings through an interactive Streamlit dashboard and communicating the key insights through the research paper and executive summary.

In [10]:
# Save final KPI summary

final_kpis.to_csv(
    "../outputs/tables/final_kpi_summary.csv",
    index=False
)

print("Final KPI table saved successfully.")

Final KPI table saved successfully.
